# SST-BR 10-second Evaluation Demo

Two consecutive local predictions are averaged. The reference is estimated **directly from the corresponding 10-second PPG samples**, not from shorter-label means.

## Build deterministic synthetic predictions and direct PPG references

In [ ]:
import numpy as np
import pandas as pd
from sstbr.demo.synthetic import generate_synthetic_recording
from sstbr.data.ppg_labels import estimate_ppg_hr
from sstbr.evaluation import aggregate_consecutive_predictions, attach_direct_references, regression_metrics

recording = generate_synthetic_recording()
direct_values = [estimate_ppg_hr(recording.ppg[i*300:(i+1)*300]) for i in range(3)]
local_predictions = [direct_values[0]+12, direct_values[0]+18,
                     direct_values[1]-15, direct_values[1]-9,
                     direct_values[2]+17, direct_values[2]+23]
local = pd.DataFrame({"fold":["synthetic"]*6, "subject_id":["synthetic"]*6,
                      "source_segment_id":["synthetic-segment"]*6,
                      "local_window_id":range(6), "pred_hr":local_predictions})
direct = pd.DataFrame({"subject_id":["synthetic"]*3,
                       "source_segment_id":["synthetic-segment"]*3,
                       "evaluation_window_id":range(3), "hr_bpm":direct_values})
evaluation = attach_direct_references(aggregate_consecutive_predictions(local), direct)
evaluation[["evaluation_window_id", "source_window_ids", "pred_hr", "gt_hr", "error"]]

## RMSE, MAE, Pearson r, and pooled rows

In [ ]:
import matplotlib.pyplot as plt
metrics = regression_metrics(evaluation.gt_hr, evaluation.pred_hr)
print({key: round(value, 3) if isinstance(value, float) else value for key, value in metrics.items()})
fig, ax = plt.subplots(figsize=(6, 3.4))
x = np.arange(len(evaluation))
ax.plot(x, evaluation.gt_hr, "o-", label="Direct 10-s PPG reference", color="#2e8b57")
ax.plot(x, evaluation.pred_hr, "s--", label="Aggregated prediction", color="#247ba0")
ax.set(xlabel="Evaluation window", ylabel="HR (BPM)", title="Synthetic 10-s evaluation")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

For multiple folds, concatenate all keyed 10-second rows before computing the pooled metrics. Per-fold metric averages are not the pooled endpoint.